# (8) Uncertainty Quantification for Motor Performance Prediction

This chapter explores uncertainty quantification techniques for motor performance prediction, focusing on methods to estimate confidence intervals, identify prediction reliability, and enable robust decision-making in motor design applications.

## Learning Objectives

- Understand different types of uncertainty in motor performance prediction
- Master Monte Carlo Dropout for Bayesian neural networks
- Implement ensemble methods for uncertainty estimation
- Learn Bayesian neural networks for probabilistic predictions
- Explore practical applications of uncertainty in motor design decisions

## 8.1 Types of Uncertainty in Motor Performance Prediction

### 8.1.1 Aleatoric vs Epistemic Uncertainty

Uncertainty in motor performance prediction can be categorized into two main types:

1. **Aleatoric Uncertainty**: Irreducible uncertainty due to inherent randomness in the system
2. **Epistemic Uncertainty**: Reducible uncertainty due to lack of knowledge or limited data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("🔍 Uncertainty Quantification for Motor Performance Prediction")
print("=" * 65)
print("🎯 Chapter Goals:")
print("  • Understand aleatoric vs epistemic uncertainty")
print("  • Implement Monte Carlo Dropout methods")
print("  • Master ensemble uncertainty estimation")
print("  • Apply Bayesian neural networks")
print("  • Enable robust motor design decisions")

In [ ]:
def uncertainty_types_demo():
    """Demonstrate different types of uncertainty in motor performance prediction"""
    
    # Generate motor performance data with different uncertainty types
    np.random.seed(42)
    speeds = np.linspace(1000, 6000, 100)
    
    # True underlying efficiency curve
    true_efficiency = 0.92 - 0.15 * (speeds / 6000)**2 - 0.05 * np.sin(speeds / 1000)
    
    # Aleatoric uncertainty (measurement noise, inherent variability)
    aleatoric_std = 0.02 + 0.01 * (speeds / 6000)  # Increases with speed
    aleatoric_noise = np.random.normal(0, aleatoric_std)
    
    # Epistemic uncertainty (model uncertainty due to limited data)
    # Simulate model bias that reduces with more data
    data_availability = np.array([50, 30, 80, 40, 20, 60])  # Number of data points at different speed ranges
    epistemic_bias = 0.05 * np.exp(-data_availability / 20)  # Higher bias with less data
    
    # Create piecewise epistemic uncertainty
    epistemic_uncertainty = np.zeros_like(speeds)
    speed_ranges = [(1000, 2000), (2000, 3000), (3000, 4000), (4000, 5000), (5000, 6000)]
    for i, (start, end) in enumerate(speed_ranges[:5]):
        mask = (speeds >= start) & (speeds <= end)
        epistemic_uncertainty[mask] = epistemic_bias[i]
    
    # Total measured efficiency
    measured_efficiency = true_efficiency + epistemic_uncertainty + aleatoric_noise
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Types of Uncertainty in Motor Performance Prediction', fontsize=16, fontweight='bold')
    
    # Plot 1: True efficiency curve
    ax1 = axes[0, 0]
    ax1.plot(speeds, true_efficiency, 'b-', linewidth=3, label='True Efficiency')
    ax1.set_xlabel('Speed (RPM)', fontweight='bold')
    ax1.set_ylabel('Efficiency', fontweight='bold')
    ax1.set_title('True Motor Efficiency', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0.6, 1.0)
    
    # Plot 2: Aleatoric uncertainty
    ax2 = axes[0, 1]
    ax2.plot(speeds, true_efficiency, 'b-', linewidth=2, label='True Efficiency')
    ax2.fill_between(speeds, 
                    true_efficiency - 2*aleatoric_std,
                    true_efficiency + 2*aleatoric_std,
                    alpha=0.3, color='blue', label='±2σ Aleatoric')
    ax2.set_xlabel('Speed (RPM)', fontweight='bold')
    ax2.set_ylabel('Efficiency', fontweight='bold')
    ax2.set_title('Aleatoric Uncertainty (Measurement Noise)', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0.6, 1.0)
    
    # Plot 3: Epistemic uncertainty
    ax3 = axes[0, 2]
    ax3.plot(speeds, true_efficiency, 'b-', linewidth=2, label='True Efficiency')
    ax3.fill_between(speeds,
                    true_efficiency - 2*epistemic_uncertainty,
                    true_efficiency + 2*epistemic_uncertainty,
                    alpha=0.3, color='red', label='±2σ Epistemic')
    ax3.set_xlabel('Speed (RPM)', fontweight='bold')
    ax3.set_ylabel('Efficiency', fontweight='bold')
    ax3.set_title('Epistemic Uncertainty (Model Uncertainty)', fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0.6, 1.0)
    
    # Plot 4: Total uncertainty
    ax4 = axes[1, 0]
    total_std = np.sqrt(aleatoric_std**2 + epistemic_uncertainty**2)
    ax4.plot(speeds, true_efficiency, 'b-', linewidth=2, label='True Efficiency')
    ax4.fill_between(speeds,
                    true_efficiency - 2*total_std,
                    true_efficiency + 2*total_std,
                    alpha=0.3, color='purple', label='±2σ Total')
    ax4.set_xlabel('Speed (RPM)', fontweight='bold')
    ax4.set_ylabel('Efficiency', fontweight='bold')
    ax4.set_title('Total Uncertainty (Combined)', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(0.6, 1.0)
    
    # Plot 5: Measured data with uncertainties
    ax5 = axes[1, 1]
    # Sample data points with uncertainties
    sample_indices = np.random.choice(len(speeds), 20, replace=False)
    sample_speeds = speeds[sample_indices]
    sample_efficiency = measured_efficiency[sample_indices]
    sample_total_std = total_std[sample_indices]
    
    ax5.errorbar(sample_speeds, sample_efficiency, yerr=2*sample_total_std,
                fmt='o', capsize=5, capthick=2, alpha=0.7, label='Measured Data')
    ax5.plot(speeds, true_efficiency, 'b--', linewidth=2, alpha=0.7, label='True Efficiency')
    ax5.set_xlabel('Speed (RPM)', fontweight='bold')
    ax5.set_ylabel('Efficiency', fontweight='bold')
    ax5.set_title('Sampled Measurements with Uncertainty', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    ax5.set_ylim(0.6, 1.0)
    
    # Plot 6: Uncertainty decomposition
    ax6 = axes[1, 2]
    ax6.plot(speeds, aleatoric_std, 'b-', linewidth=2, label='Aleatoric')
    ax6.plot(speeds, epistemic_uncertainty, 'r-', linewidth=2, label='Epistemic')
    ax6.plot(speeds, total_std, 'k-', linewidth=2, label='Total')
    ax6.set_xlabel('Speed (RPM)', fontweight='bold')
    ax6.set_ylabel('Standard Deviation', fontweight='bold')
    ax6.set_title('Uncertainty Components', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print insights
    print("🔍 Uncertainty Analysis:")
    print("=" * 40)
    print(f"📊 Average aleatoric uncertainty: {np.mean(aleatoric_std):.4f}")
    print(f"🧠 Average epistemic uncertainty: {np.mean(epistemic_uncertainty):.4f}")
    print(f"⚖️ Average total uncertainty: {np.mean(total_std):.4f}")
    print(f"📈 Epistemic/Aleatoric ratio: {np.mean(epistemic_uncertainty)/np.mean(aleatoric_std):.2f}")
    print("\n💡 Key Insights:")
    print("  • Aleatoric uncertainty is inherent and cannot be reduced")
    print("  • Epistemic uncertainty can be reduced with more data")
    print("  • Total uncertainty combines both types quadratically")
    print("  • Uncertainty quantification enables reliable decision-making")
    
    return {
        'true_efficiency': true_efficiency,
        'measured_efficiency': measured_efficiency,
        'aleatoric_std': aleatoric_std,
        'epistemic_uncertainty': epistemic_uncertainty,
        'total_std': total_std
    }

# Run uncertainty types demonstration
uncertainty_data = uncertainty_types_demo()

## 8.2 Monte Carlo Dropout for Bayesian Approximation

Monte Carlo Dropout provides a practical approach to approximate Bayesian inference in neural networks by performing multiple forward passes with dropout enabled during inference.

In [ ]:
class MCDropoutMotorModel(nn.Module):
    """Motor performance prediction model with Monte Carlo Dropout"""
    
    def __init__(self, input_dim=4, hidden_dims=[64, 32], output_dim=3, dropout_rate=0.1):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim
        
        # Output layer with heteroscedastic uncertainty
        self.feature_extractor = nn.Sequential(*layers)
        self.mean_layer = nn.Linear(prev_dim, output_dim)
        self.log_var_layer = nn.Linear(prev_dim, output_dim)  # Learn variance
        
        self.dropout_rate = dropout_rate
        
    def forward(self, x, return_uncertainty=False):
        features = self.feature_extractor(x)
        mean = self.mean_layer(features)
        
        if return_uncertainty:
            log_var = self.log_var_layer(features)
            # Clamp log variance for numerical stability
            log_var = torch.clamp(log_var, min=-10, max=2)
            return mean, log_var
        else:
            return mean
    
    def predict_with_uncertainty(self, x, n_samples=100):
        """Make predictions with Monte Carlo dropout"""
        self.train()  # Enable dropout during inference
        
        predictions = []
        log_vars = []
        
        with torch.no_grad():
            for _ in range(n_samples):
                mean, log_var = self.forward(x, return_uncertainty=True)
                predictions.append(mean)
                log_vars.append(log_var)
        
        self.eval()  # Return to evaluation mode
        
        # Stack predictions
        predictions = torch.stack(predictions)  # (n_samples, batch_size, output_dim)
        log_vars = torch.stack(log_vars)  # (n_samples, batch_size, output_dim)
        
        # Calculate statistics
        mean_pred = predictions.mean(dim=0)  # (batch_size, output_dim)
        
        # Total uncertainty = aleatoric + epistemic
        aleatoric_var = torch.exp(log_vars).mean(dim=0)  # Mean of learned variances
        epistemic_var = predictions.var(dim=0)  # Variance across MC samples
        total_var = aleatoric_var + epistemic_var
        
        total_std = torch.sqrt(total_var)
        
        return {
            'mean': mean_pred,
            'std': total_std,
            'aleatoric_std': torch.sqrt(aleatoric_var),
            'epistemic_std': torch.sqrt(epistemic_var),
            'all_predictions': predictions
        }

def generate_motor_data_with_uncertainty(num_samples=1000, seed=42):
    """Generate motor performance data with realistic uncertainties"""
    
    np.random.seed(seed)
    
    # Operating conditions
    speeds = np.random.uniform(1000, 6000, num_samples)
    torques = np.random.uniform(20, 200, num_samples)
    currents = np.random.uniform(10, 150, num_samples)
    temperatures = np.random.uniform(20, 80, num_samples)
    
    # True underlying performance
    efficiency = 0.9 - 0.15 * (speeds / 6000)**2 - 0.05 * (torques / 200)**2
    power_loss = 0.1 * (currents / 100)**2 * (1 + 0.01 * (temperatures - 50))
    thermal_rise = power_loss * 40 / (1 + 0.01 * speeds)
    
    # Add heteroscedastic noise (variance depends on input)
    eff_noise_std = 0.02 + 0.01 * (speeds / 6000)
    power_noise_std = 0.005 + 0.002 * (currents / 100)
    thermal_noise_std = 1.0 + 0.5 * (temperatures / 80)
    
    efficiency += np.random.normal(0, eff_noise_std)
    power_loss += np.random.normal(0, power_noise_std)
    thermal_rise += np.random.normal(0, thermal_noise_std)
    
    # Input features
    X = np.column_stack([speeds, torques, currents, temperatures])
    
    # Output targets
    y = np.column_stack([efficiency, power_loss, thermal_rise])
    
    return X, y

def demonstrate_mc_dropout():
    """Demonstrate Monte Carlo Dropout for uncertainty quantification"""
    
    # Generate data
    print("🔄 Generating Motor Performance Data...")
    X_train, y_train = generate_motor_data_with_uncertainty(800, seed=42)
    X_test, y_test = generate_motor_data_with_uncertainty(200, seed=123)
    
    print(f"✅ Training data: {X_train.shape}")
    print(f"✅ Test data: {X_test.shape}")
    
    # Normalize data
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    X_train_scaled = scaler_X.fit_transform(X_train)
    y_train_scaled = scaler_y.fit_transform(y_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_test_scaled = scaler_y.transform(y_test)
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train_scaled)
    y_train_tensor = torch.FloatTensor(y_train_scaled)
    X_test_tensor = torch.FloatTensor(X_test_scaled)
    y_test_tensor = torch.FloatTensor(y_test_scaled)
    
    # Create datasets and loaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    # Initialize model
    model = MCDropoutMotorModel(
        input_dim=4,
        hidden_dims=[64, 32],
        output_dim=3,
        dropout_rate=0.2
    )
    
    # Training
    print("\n🏋️ Training MC Dropout Model...")
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    train_losses = []
    
    for epoch in range(100):
        model.train()
        total_loss = 0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            mean_pred, log_var = model(batch_X, return_uncertainty=True)
            
            # Negative log likelihood loss for heteroscedastic uncertainty
            loss = 0.5 * torch.exp(-log_var) * (batch_y - mean_pred)**2 + 0.5 * log_var
            loss = loss.mean()
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        if epoch % 20 == 0:
            print(f"  Epoch {epoch}: Loss = {avg_loss:.4f}")
    
    # Prediction with uncertainty
    print("\n🔍 Making Predictions with Uncertainty...")
    n_mc_samples = 100
    predictions = model.predict_with_uncertainty(X_test_tensor, n_samples=n_mc_samples)
    
    # Convert back to original scale
    mean_pred_orig = scaler_y.inverse_transform(predictions['mean'].numpy())
    std_pred_orig = scaler_y.inverse_transform(
        np.concatenate([predictions['mean'].numpy(), predictions['std'].numpy()], axis=1)
    )[:, 3:]  # Extract std part
    
    aleatoric_std_orig = scaler_y.inverse_transform(
        np.concatenate([predictions['mean'].numpy(), predictions['aleatoric_std'].numpy()], axis=1)
    )[:, 3:]
    
    epistemic_std_orig = scaler_y.inverse_transform(
        np.concatenate([predictions['mean'].numpy(), predictions['epistemic_std'].numpy()], axis=1)
    )[:, 3:]
    
    # Visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Monte Carlo Dropout for Uncertainty Quantification', fontsize=16, fontweight='bold')
    
    metrics = ['Efficiency', 'Power Loss', 'Thermal Rise']
    colors = ['b', 'r', 'g']
    
    # Plot 1-3: Prediction with uncertainty bands
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        ax = axes[0, i]
        
        # Sort by speed for better visualization
        sort_idx = np.argsort(X_test[:, 0])
        speeds_sorted = X_test[sort_idx, 0]
        y_sorted = y_test[sort_idx, i]
        mean_sorted = mean_pred_orig[sort_idx, i]
        std_sorted = std_pred_orig[sort_idx, i]
        
        ax.plot(speeds_sorted, mean_sorted, color=color, linewidth=2, label='Predicted Mean')
        ax.fill_between(speeds_sorted,
                       mean_sorted - 2*std_sorted,
                       mean_sorted + 2*std_sorted,
                       alpha=0.3, color=color, label='±2σ Confidence')
        ax.scatter(speeds_sorted, y_sorted, alpha=0.5, s=10, color='black', label='True Values')
        
        ax.set_xlabel('Speed (RPM)', fontweight='bold')
        ax.set_ylabel(metric, fontweight='bold')
        ax.set_title(f'{metric} Prediction with Uncertainty', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Plot 4: Uncertainty decomposition
    ax4 = axes[1, 0]
    metric_idx = 0  # Efficiency
    
    sort_idx = np.argsort(X_test[:, 0])
    speeds_sorted = X_test[sort_idx, 0]
    aleatoric_sorted = aleatoric_std_orig[sort_idx, metric_idx]
    epistemic_sorted = epistemic_std_orig[sort_idx, metric_idx]
    total_sorted = std_pred_orig[sort_idx, metric_idx]
    
    ax4.plot(speeds_sorted, aleatoric_sorted, 'b-', linewidth=2, label='Aleatoric')
    ax4.plot(speeds_sorted, epistemic_sorted, 'r-', linewidth=2, label='Epistemic')
    ax4.plot(speeds_sorted, total_sorted, 'k-', linewidth=2, label='Total')
    ax4.set_xlabel('Speed (RPM)', fontweight='bold')
    ax4.set_ylabel('Standard Deviation', fontweight='bold')
    ax4.set_title('Uncertainty Decomposition (Efficiency)', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # Plot 5: Calibration plot
    ax5 = axes[1, 1]
    # Calculate prediction intervals coverage
    confidence_levels = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
    empirical_coverages = []
    
    for confidence in confidence_levels:
        z_score = stats.norm.ppf((1 + confidence) / 2)
        coverage = 0
        
        for i in range(len(y_test)):
            mean_val = mean_pred_orig[i, 0]  # Efficiency
            std_val = std_pred_orig[i, 0]
            true_val = y_test[i, 0]
            
            if abs(true_val - mean_val) <= z_score * std_val:
                coverage += 1
        
        empirical_coverages.append(coverage / len(y_test))
    
    ax5.plot(confidence_levels, confidence_levels, 'k--', label='Perfect Calibration')
    ax5.plot(confidence_levels, empirical_coverages, 'bo-', linewidth=2, markersize=6, label='MC Dropout')
    ax5.set_xlabel('Confidence Level', fontweight='bold')
    ax5.set_ylabel('Empirical Coverage', fontweight='bold')
    ax5.set_title('Calibration Plot', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    ax5.set_xlim(0.4, 1.0)
    ax5.set_ylim(0.4, 1.0)
    
    # Plot 6: Sample MC dropout predictions
    ax6 = axes[1, 2]
    sample_idx = 0
    all_predictions = predictions['all_predictions'][:, sample_idx, 0].numpy()  # Efficiency
    true_value = y_test[sample_idx, 0]
    
    ax6.hist(all_predictions, bins=30, alpha=0.7, color='skyblue', edgecolor='black',
            label=f'MC Samples (mean={np.mean(all_predictions):.3f})')
    ax6.axvline(x=true_value, color='red', linestyle='--', linewidth=2,
               label=f'True Value ({true_value:.3f})')
    ax6.axvline(x=np.mean(all_predictions), color='blue', linestyle='-', linewidth=2,
               label=f'MC Mean ({np.mean(all_predictions):.3f})')
    ax6.set_xlabel('Predicted Efficiency', fontweight='bold')
    ax6.set_ylabel('Frequency', fontweight='bold')
    ax6.set_title(f'MC Dropout Distribution\n(Sample {sample_idx+1})', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print("\n📊 Monte Carlo Dropout Results:")
    print("=" * 50)
    print(f"🎯 MC samples: {n_mc_samples}")
    print(f"📈 Average aleatoric std: {np.mean(aleatoric_std_orig):.4f}")
    print(f"🧠 Average epistemic std: {np.mean(epistemic_std_orig):.4f}")
    print(f"⚖️ Average total std: {np.mean(std_pred_orig):.4f}")
    print(f"🎯 MSE (point estimate): {mean_squared_error(y_test, mean_pred_orig):.4f}")
    print(f"📊 Coverage at 95% confidence: {empirical_coverages[-1]:.3f}")
    
    return {
        'model': model,
        'predictions': predictions,
        'mean_pred_orig': mean_pred_orig,
        'std_pred_orig': std_pred_orig,
        'calibration': empirical_coverages
    }

# Run MC Dropout demonstration
mc_results = demonstrate_mc_dropout()

## 8.3 Ensemble Methods for Uncertainty Estimation

Ensemble methods train multiple models with different initializations and architectures, then combine their predictions to estimate uncertainty through model disagreement.

In [ ]:
class EnsembleMotorModel:
    """Ensemble of motor performance models for uncertainty estimation"""
    
    def __init__(self, n_models=5, input_dim=4, hidden_dims=None, output_dim=3):
        self.n_models = n_models
        self.models = []
        
        for i in range(n_models):
            # Create diverse architectures
            if hidden_dims is None:
                if i % 3 == 0:
                    hidden = [64, 32]
                elif i % 3 == 1:
                    hidden = [48, 24, 12]
                else:
                    hidden = [80, 40]
            else:
                hidden = hidden_dims
            
            model = self._create_single_model(input_dim, hidden, output_dim)
            self.models.append(model)
    
    def _create_single_model(self, input_dim, hidden_dims, output_dim):
        """Create a single neural network model"""
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        
        return nn.Sequential(*layers)
    
    def train_models(self, X_train, y_train, epochs=100, batch_size=32):
        """Train all models in the ensemble"""
        train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
        
        for i, model in enumerate(self.models):
            print(f"Training model {i+1}/{self.n_models}...")
            
            # Different random seed for each model
            torch.manual_seed(42 + i)
            np.random.seed(42 + i)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            criterion = nn.MSELoss()
            
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            
            for epoch in range(epochs):
                model.train()
                total_loss = 0
                
                for batch_X, batch_y in train_loader:
                    optimizer.zero_grad()
                    predictions = model(batch_X)
                    loss = criterion(predictions, batch_y)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
                
                if epoch % 20 == 0 and i == 0:  # Only print for first model
                    print(f"  Epoch {epoch}: Loss = {total_loss/len(train_loader):.4f}")
    
    def predict_with_uncertainty(self, X):
        """Make predictions with uncertainty estimation"""
        X_tensor = torch.FloatTensor(X)
        
        predictions = []
        
        for model in self.models:
            model.eval()
            with torch.no_grad():
                pred = model(X_tensor).numpy()
                predictions.append(pred)
        
        predictions = np.array(predictions)  # (n_models, n_samples, output_dim)
        
        # Calculate ensemble statistics
        mean_pred = predictions.mean(axis=0)
        std_pred = predictions.std(axis=0)
        
        return {
            'mean': mean_pred,
            'std': std_pred,
            'all_predictions': predictions
        }

def demonstrate_ensemble_uncertainty():
    """Demonstrate ensemble methods for uncertainty estimation"""
    
    # Generate data
    print("🔄 Generating Data for Ensemble Training...")
    X_train, y_train = generate_motor_data_with_uncertainty(1000, seed=42)
    X_test, y_test = generate_motor_data_with_uncertainty(200, seed=123)
    
    # Normalize data
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    X_train_scaled = scaler_X.fit_transform(X_train)
    y_train_scaled = scaler_y.fit_transform(y_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_test_scaled = scaler_y.transform(y_test)
    
    # Create and train ensemble
    print("\n🏗️ Creating and Training Ensemble...")
    ensemble = EnsembleMotorModel(n_models=7, input_dim=4, output_dim=3)
    ensemble.train_models(X_train_scaled, y_train_scaled, epochs=80)
    
    # Make predictions with uncertainty
    print("\n🔍 Making Ensemble Predictions...")
    ensemble_predictions = ensemble.predict_with_uncertainty(X_test_scaled)
    
    # Convert back to original scale
    mean_pred_orig = scaler_y.inverse_transform(ensemble_predictions['mean'])
    std_pred_orig = scaler_y.inverse_transform(
        np.concatenate([ensemble_predictions['mean'], ensemble_predictions['std']], axis=1)
    )[:, 3:]  # Extract std part
    
    # Visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Ensemble Methods for Uncertainty Estimation', fontsize=16, fontweight='bold')
    
    metrics = ['Efficiency', 'Power Loss', 'Thermal Rise']
    colors = ['b', 'r', 'g']
    
    # Plot 1-3: Ensemble predictions with uncertainty
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        ax = axes[0, i]
        
        # Sort by speed for better visualization
        sort_idx = np.argsort(X_test[:, 0])
        speeds_sorted = X_test[sort_idx, 0]
        y_sorted = y_test[sort_idx, i]
        mean_sorted = mean_pred_orig[sort_idx, i]
        std_sorted = std_pred_orig[sort_idx, i]
        
        ax.plot(speeds_sorted, mean_sorted, color=color, linewidth=2, label='Ensemble Mean')
        ax.fill_between(speeds_sorted,
                       mean_sorted - 2*std_sorted,
                       mean_sorted + 2*std_sorted,
                       alpha=0.3, color=color, label='±2σ Ensemble')
        ax.scatter(speeds_sorted, y_sorted, alpha=0.5, s=10, color='black', label='True Values')
        
        ax.set_xlabel('Speed (RPM)', fontweight='bold')
        ax.set_ylabel(metric, fontweight='bold')
        ax.set_title(f'{metric} Ensemble Prediction', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Plot 4: Individual model predictions
    ax4 = axes[1, 0]
    metric_idx = 0  # Efficiency
    
    sort_idx = np.argsort(X_test[:, 0])
    speeds_sorted = X_test[sort_idx, 0]
    
    # Show predictions from individual models
    for i in range(min(5, ensemble.n_models)):  # Show first 5 models
        model_pred = scaler_y.inverse_transform(
            ensemble_predictions['all_predictions'][i]
        )[sort_idx, metric_idx]
        ax4.plot(speeds_sorted, model_pred, alpha=0.6, linewidth=1, label=f'Model {i+1}')
    
    # Show ensemble mean
    ax4.plot(speeds_sorted, mean_pred_orig[sort_idx, metric_idx], 'k-', 
            linewidth=3, label='Ensemble Mean')
    
    ax4.set_xlabel('Speed (RPM)', fontweight='bold')
    ax4.set_ylabel('Efficiency', fontweight='bold')
    ax4.set_title('Individual Model Predictions', fontweight='bold')
    ax4.legend(fontsize=8)
    ax4.grid(True, alpha=0.3)
    
    # Plot 5: Uncertainty vs data density
    ax5 = axes[1, 1]
    # Calculate local data density
    from sklearn.neighbors import NearestNeighbors
    nbrs = NearestNeighbors(n_neighbors=10).fit(X_train)
    distances, _ = nbrs.kneighbors(X_test)
    local_density = 1 / np.mean(distances, axis=1)
    
    # Plot uncertainty vs density
    uncertainty_efficiency = std_pred_orig[:, 0]
    ax5.scatter(local_density, uncertainty_efficiency, alpha=0.6)
    ax5.set_xlabel('Local Data Density', fontweight='bold')
    ax5.set_ylabel('Efficiency Uncertainty', fontweight='bold')
    ax5.set_title('Uncertainty vs Data Density', fontweight='bold')
    ax5.grid(True, alpha=0.3)
    
    # Add correlation
    correlation = np.corrcoef(local_density, uncertainty_efficiency)[0, 1]
    ax5.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
            transform=ax5.transAxes, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Plot 6: Ensemble diversity analysis
    ax6 = axes[1, 2]
    # Calculate pairwise disagreement between models
    all_predictions = ensemble_predictions['all_predictions']  # (n_models, n_samples, output_dim)
    
    disagreements = []
    for i in range(ensemble.n_models):
        for j in range(i+1, ensemble.n_models):
            disagreement = np.mean((all_predictions[i] - all_predictions[j])**2, axis=1)
            disagreements.append(disagreement)
    
    avg_disagreement = np.mean(disagreements, axis=0)
    
    # Plot disagreement vs uncertainty
    ax6.scatter(avg_disagreement, np.mean(std_pred_orig**2, axis=1), alpha=0.6)
    ax6.set_xlabel('Model Disagreement', fontweight='bold')
    ax6.set_ylabel('Prediction Variance', fontweight='bold')
    ax6.set_title('Ensemble Diversity Analysis', fontweight='bold')
    ax6.grid(True, alpha=0.3)
    
    # Add correlation
    corr_disagreement = np.corrcoef(avg_disagreement, np.mean(std_pred_orig**2, axis=1))[0, 1]
    ax6.text(0.05, 0.95, f'Correlation: {corr_disagreement:.3f}', 
            transform=ax6.transAxes, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print("\n📊 Ensemble Method Results:")
    print("=" * 50)
    print(f"🤖 Number of models: {ensemble.n_models}")
    print(f"📈 Average prediction std: {np.mean(std_pred_orig):.4f}")
    print(f"🎯 MSE (ensemble mean): {mean_squared_error(y_test, mean_pred_orig):.4f}")
    print(f"📊 Uncertainty-density correlation: {correlation:.3f}")
    print(f"🔄 Disagreement-variance correlation: {corr_disagreement:.3f}")
    
    # Compare with single model
    single_model_pred = ensemble_predictions['all_predictions'][0]
    single_mse = mean_squared_error(y_test, scaler_y.inverse_transform(single_model_pred))
    print(f"\n🚀 Performance Improvement:")
    print(f"  Ensemble vs Single Model MSE: {(1 - mean_squared_error(y_test, mean_pred_orig)/single_mse)*100:.1f}% improvement")
    
    return {
        'ensemble': ensemble,
        'predictions': ensemble_predictions,
        'mean_pred_orig': mean_pred_orig,
        'std_pred_orig': std_pred_orig
    }

# Run ensemble uncertainty demonstration
ensemble_results = demonstrate_ensemble_uncertainty()

## 8.4 Practical Applications and Decision Making

Uncertainty quantification enables more robust decision-making in motor design by identifying high-risk predictions and guiding data collection efforts.

In [ ]:
def uncertainty_driven_decisions():
    """Demonstrate uncertainty-driven decision making in motor design"""
    
    # Create decision scenarios
    print("🎯 Uncertainty-Driven Decision Making Scenarios")
    print("=" * 55)
    
    # Scenario 1: Motor selection with uncertainty
    print("\n📊 Scenario 1: Motor Selection with Uncertainty")
    print("-" * 40)
    
    # Simulate motor options with predictions and uncertainties
    motor_options = {
        'IPM-50kW': {'efficiency': 0.92, 'std': 0.015, 'cost': 1000},
        'IPM-75kW': {'efficiency': 0.90, 'std': 0.025, 'cost': 1500},
        'FSCW-50kW': {'efficiency': 0.88, 'std': 0.035, 'cost': 800},
        'FSCW-75kW': {'efficiency': 0.86, 'std': 0.045, 'cost': 1200}
    }
    
    # Decision criteria with uncertainty consideration
    efficiency_threshold = 0.89
    uncertainty_penalty = 2.0  # Weight for uncertainty
    
    print(f"Efficiency requirement: ≥{efficiency_threshold}")
    print(f"Uncertainty penalty factor: {uncertainty_penalty}")
    print("\nMotor Evaluation:")
    
    best_motor = None
    best_score = -np.inf
    
    for motor, specs in motor_options.items():
        eff = specs['efficiency']
        std = specs['std']
        cost = specs['cost']
        
        # Probability of meeting efficiency requirement
        prob_meeting_req = 1 - stats.norm.cdf(efficiency_threshold, eff, std)
        
        # Utility score considering efficiency, uncertainty, and cost
        utility_score = eff - uncertainty_penalty * std - cost / 10000
        
        print(f"  {motor}:")
        print(f"    Efficiency: {eff:.3f} ± {std:.3f}")
        print(f"    P(eff ≥ {efficiency_threshold}): {prob_meeting_req:.3f}")
        print(f"    Utility score: {utility_score:.3f}")
        print(f"    Cost: ${cost}")
        
        if prob_meeting_req > 0.7 and utility_score > best_score:
            best_score = utility_score
            best_motor = motor
    
    print(f"\n🏆 Recommended motor: {best_motor} (score: {best_score:.3f})")
    
    # Scenario 2: Data collection prioritization
    print("\n📊 Scenario 2: Data Collection Prioritization")
    print("-" * 40)
    
    # Simulate operating regions with different uncertainties
    operating_regions = [
        {'speed_range': (1000, 2000), 'torque_range': (20, 80), 'uncertainty': 0.02, 'priority': 'low'},
        {'speed_range': (2000, 3000), 'torque_range': (80, 120), 'uncertainty': 0.05, 'priority': 'medium'},
        {'speed_range': (4000, 5000), 'torque_range': (120, 160), 'uncertainty': 0.08, 'priority': 'high'},
        {'speed_range': (5000, 6000), 'torque_range': (160, 200), 'uncertainty': 0.12, 'priority': 'critical'}
    ]
    
    print("Operating regions ranked by uncertainty:")
    sorted_regions = sorted(operating_regions, key=lambda x: x['uncertainty'], reverse=True)
    
    for i, region in enumerate(sorted_regions, 1):
        speed_range = region['speed_range']
        torque_range = region['torque_range']
        uncertainty = region['uncertainty']
        priority = region['priority']
        
        print(f"  {i}. Speed: {speed_range[0]}-{speed_range[1]} RPM, "
              f"Torque: {torque_range[0]}-{torque_range[1]} Nm")
        print(f"     Uncertainty: {uncertainty:.3f}, Priority: {priority}")
    
    # Scenario 3: Risk-aware performance guarantees
    print("\n📊 Scenario 3: Risk-Aware Performance Guarantees")
    print("-" * 40)
    
    # Simulate performance predictions with confidence intervals
    performance_targets = {
        'Efficiency': {'target': 0.90, 'confidence': 0.95},
        'Power_Loss': {'target': 15.0, 'confidence': 0.90},
        'Thermal_Rise': {'target': 40.0, 'confidence': 0.99}
    }
    
    predictions = {
        'Efficiency': {'mean': 0.92, 'std': 0.018},
        'Power_Loss': {'mean': 12.5, 'std': 2.1},
        'Thermal_Rise': {'mean': 38.2, 'std': 4.8}
    }
    
    print("Performance guarantee analysis:")
    
    guarantees_met = 0
    total_metrics = len(performance_targets)
    
    for metric, target in performance_targets.items():
        pred = predictions[metric]
        target_val = target['target']
        confidence = target['confidence']
        
        if metric == 'Efficiency':
            # For efficiency, we want P(efficiency >= target)
            z_score = stats.norm.ppf(1 - confidence)
            lower_bound = pred['mean'] + z_score * pred['std']
            guarantee_met = lower_bound >= target_val
            comparison = "≥"
        else:
            # For power loss and thermal rise, we want P(metric <= target)
            z_score = stats.norm.ppf(confidence)
            upper_bound = pred['mean'] + z_score * pred['std']
            guarantee_met = upper_bound <= target_val
            comparison = "≤"
        
        if guarantee_met:
            guarantees_met += 1
        
        status = "✅ MET" if guarantee_met else "❌ NOT MET"
        
        print(f"  {metric}:")
        print(f"    Target: {target_val} {comparison} (confidence: {confidence:.0%})")
        print(f"    Prediction: {pred['mean']:.3f} ± {pred['std']:.3f}")
        print(f"    Guarantee: {status}")
    
    guarantee_rate = guarantees_met / total_metrics
    print(f"\n📈 Overall guarantee rate: {guarantee_rate:.1%}")
    
    if guarantee_rate >= 0.8:
        print("🏆 Motor design meets risk requirements!")
    else:
        print("⚠️ Motor design needs improvement to meet risk requirements.")
    
    return {
        'motor_selection': best_motor,
        'data_priorities': sorted_regions,
        'guarantee_rate': guarantee_rate
    }

# Run uncertainty-driven decision making demonstration
decision_results = uncertainty_driven_decisions()

## Chapter Summary

### Key Takeaways

1. **Uncertainty Types**: Understanding the distinction between aleatoric (irreducible) and epistemic (reducible) uncertainty is crucial for appropriate uncertainty quantification strategies.

2. **Monte Carlo Dropout**: Provides a practical approach to Bayesian approximation by performing multiple stochastic forward passes during inference.

3. **Ensemble Methods**: Model ensembles capture uncertainty through disagreement between diverse models trained with different initializations and architectures.

4. **Practical Decision Making**: Uncertainty quantification enables risk-aware motor selection, data collection prioritization, and performance guarantee specification.

### Implementation Guidelines

- **Choose appropriate method**: MC Dropout for efficiency, ensembles for robustness
- **Calibrate uncertainties**: Ensure predicted uncertainties match observed frequencies
- **Consider heteroscedasticity**: Allow uncertainty to vary with input conditions
- **Validate uncertainty quality**: Use proper scoring rules and calibration plots

### Next Steps

In the next chapter, we will explore results analysis and visualization techniques for motor performance prediction, focusing on effective communication of model performance and insights.